<font face="Times New Roman" size=5>
<div dir=rtl align="center">
<font face="Times New Roman" size=5>
In The Name of God
</font>
<br>
<img src="https://logoyar.com/content/wp-content/uploads/2021/04/sharif-university-logo.png" alt="University Logo" width="150" height="150">
<br>
<font face="Times New Roman" size=4 align=center>
Sharif University of Technology - Department of Electrical Engineering
</font>
<br>
<font color="#008080" size=6>
Deep Generative Models
</font>
<hr/>
<font color="#800080" size=5>
Assignment 1 : Deep Autoregressive Models
<br>
</font>
<font size=5>
Instructor: Dr. S. Amini
<br>
</font>
<font size=4>
Fall 2025
<br>
</font>
<font face="Times New Roman" size=4>
</font>
<hr>
<font color='red'  size=4>
<br>
</font>
<font face="Times New Roman" size=4 align=center>
Feel free to ask your questions in Telegram : @imoonamm
</font>
<br>
<hr>
</div></font>

Amir Kooshan Fattah Hesari - 401102191

**You should only change the blank sections, marked with TODO**

Pay attention to docstrings, as they may drastically help with your implementation.

You are advised to read all related papers and material, to help you better understand the design of each model.

# Question 2: Pixel CNN / Pixel RNN (Optional)

In this question, we explore **deep autoregressive models** for image generation, focusing on PixelCNN and PixelRNN.  

Natural images are high-dimensional and exhibit strong spatial correlations between neighboring pixels. The **PixelRNN** aims to model the **full joint distribution** of pixel intensities, enabling exact likelihood computation and realistic image generation.  

Unlike VAEs or GANs, which rely on latent variables or adversarial training, PixelRNN treats the **image as a sequence of pixels** and directly models:  

$$
p(\mathbf{x}) = \prod_{i=1}^{N} p(x_i \mid x_1, x_2, \ldots, x_{i-1})
$$

Here, each pixel \(x_i\) is conditioned on all previously generated pixels, following a fixed **raster-scan order** (top-left → bottom-right). This transforms image modeling into a **sequence modeling task**, allowing recurrent neural networks (RNNs) to capture long-range spatial dependencies.  

---

### Core Idea

PixelRNN introduces **two-dimensional LSTM architectures** to preserve autoregressive dependencies in images:

- **Row LSTM:** Processes the image row by row (left-to-right), with recurrent connections across both rows and columns.  
- **Diagonal Bi-LSTM:** Processes pixels along diagonals, increasing the receptive field to include all previously generated pixels.  

To maintain causality across color channels, each pixel’s RGB values are generated sequentially (R → G → B).  

![PixelRNN Architecture](image.png)

---

### Architectural Components

1. **Masked Convolutions**  
   Convolutional filters are masked (type A or B) to prevent access to future pixels or channels during training.  

2. **Residual Connections**  
   Stacked recurrent layers use residual links to facilitate gradient flow in deep networks.  

3. **Discrete Softmax Output**  
   Each pixel channel is modeled as a categorical distribution over 256 intensity values, allowing exact log-likelihood computation.  

4. **Loss Function**  
   The training objective is the **negative log-likelihood** of the true pixels under the predicted distributions:

$$
\mathcal{L} = -\sum_i \log p(x_i \mid x_{<i})
$$

---

### Sampling and Generation

Image generation in PixelRNN is **fully autoregressive**:

1. Start at the top-left pixel.  
2. Sequentially sample each pixel from its predicted conditional distribution.  
3. Repeat until the entire image is generated.  

This method produces coherent, high-quality images but is computationally intensive, as each pixel depends on all previously generated pixels.  

---

**Reference:** [PixelRNN Paper (2016)](https://arxiv.org/abs/1601.06759)


## load Cifar10

5 Points

In [1]:
import os
import math
import struct
from typing import List, Tuple
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as F

# Filled-in constants for CIFAR-10
IMAGE_SIZE = 24                  # 32x32 -> 24x24 (as mentioned in _eval_transform)
NUM_CLASSES = 10                 # CIFAR-10 has 10 classes
NUM_EXAMPLES_PER_EPOCH_FOR_TRAIN = 50000  # 5 train batches * 10,000 images
NUM_EXAMPLES_PER_EPOCH_FOR_EVAL = 10000   # 1 test batch * 10,000 images


def _read_cifar10_binary_file(path: str) -> List[Tuple[int, np.ndarray]]:
    """
    Reads a single CIFAR-10 binary file and returns a list of (label, image_array)
    where image_array is HxWxC uint8 (32x32x3).
    CIFAR-10 binary format: 1 byte label followed by 3072 bytes image (R(1024), G(1024), B(1024))
    """
    records = []
    record_size = 1 + 32 * 32 * 3
    with open(path, 'rb') as f:
        while True:
            bytes_read = f.read(record_size)
            if not bytes_read:
                break
            if len(bytes_read) != record_size:
                raise ValueError(f"Unexpected record size in {path}: {len(bytes_read)} bytes")
            # first byte is label
            label = bytes_read[0]
            img_flat = np.frombuffer(bytes_read[1:], dtype=np.uint8)
            # image is stored as [R..1024, G..1024, B..1024] each row-major for 32x32
            # reshape to (3, 32, 32) then transpose to (32, 32, 3)
            depth_major = img_flat.reshape((3, 32, 32))
            img = np.transpose(depth_major, (1, 2, 0))  # H, W, C
            records.append((int(label), img))
    return records

class CIFAR10BinaryDataset(Dataset):
    """
    Dataset that reads the CIFAR-10 binary files into memory (list of (label, image)).
    Applies transforms (train/eval) provided in init.
    """
    def __init__(self, data_dir: str, files: List[str], transform=None):
        """
        data_dir: directory containing the CIFAR-10 binary files.
        files: list of filenames (basename) to read from data_dir.
        transform: torchvision transform to apply to PIL image (or custom transform).
        """
        self.data_dir = data_dir
        self.files = files
        self.transform = transform

        # Load all records into memory (50k images is fine)
        self.records: List[Tuple[int, np.ndarray]] = []
        for fname in files:
            path = os.path.join(data_dir, fname)
            if not os.path.exists(path):
                raise ValueError(f"Failed to find file: {path}")
            self.records.extend(_read_cifar10_binary_file(path))

        if len(self.records) == 0:
            raise ValueError("No records loaded from CIFAR files.")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        label, img_np = self.records[idx]
        # Convert to PIL Image (expects HxW or HxWxC)
        img = Image.fromarray(img_np)  # mode='RGB'
        if self.transform:
            img = self.transform(img)
        # After transforms, we expect a torch.FloatTensor image of shape [C, H, W]
        # Return label as long tensor
        return img, torch.tensor(label, dtype=torch.long)


class PerImageStandardization:
    """
    Mimics TensorFlow's per_image_standardization:
      (x - mean) / max(stddev, 1.0/sqrt(N))
    Works on a torch tensor image with shape (C, H, W) and dtype=float32 in range [0,1] or raw values.
    We'll expect inputs are float tensors in range [0,1].
    """
    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        # tensor: C x H x W
        if not torch.is_floating_point(tensor):
            tensor = tensor.float()
        mean = tensor.mean()
        std = tensor.std()
        # N = number of pixels * channels
        N = tensor.numel()
        std_min = 1.0 / math.sqrt(N)
        std_adj = max(std.item(), std_min)
        return (tensor - mean) / std_adj


def _train_transform():
    """
    Returns torchvision transform for training (distortions similar to TF code):
    - Random crop to IMAGE_SIZE x IMAGE_SIZE from 32x32 image
    - Random horizontal flip
    - Random brightness and contrast (approximate TF random_brightness/random_contrast)
    - Convert to tensor (0..1)
    - Per-image standardization
    """
    # ColorJitter's brightness and contrast ranges take a factor. TF used:
    # random_brightness with max_delta=63 (on [0,255]) => approx +/- 0.25 in normalized 0..1
    # random_contrast lower=0.2 upper=1.8 => matches contrast jitter factor
    color_jitter = transforms.ColorJitter(brightness=0.25, contrast=(0.2, 1.8))
    return transforms.Compose([
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        color_jitter,
        transforms.ToTensor(),  # gives C x H x W, float in [0,1]
        PerImageStandardization(),
    ])


def _eval_transform():
    """
    Returns transform for evaluation:
     - center crop / pad to IMAGE_SIZE x IMAGE_SIZE (TF used resize_image_with_crop_or_pad which crops center)
     - Convert to tensor and per-image standardize
    """
    # For 32->24 center crop:
    return transforms.Compose([
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        PerImageStandardization(),
    ])


def get_train_loader(data_dir: str, batch_size: int, num_workers: int = 16, shuffle: bool = True):
    """
    Returns a DataLoader for training (reads data_batch_1..5).
    """
    train_files = [
        "data_batch_1.bin",
        "data_batch_2.bin",
        "data_batch_3.bin",
        "data_batch_4.bin",
        "data_batch_5.bin",
    ]
    dataset = CIFAR10BinaryDataset(
        data_dir=data_dir,
        files=train_files,
        transform=_train_transform(),
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
    )
    return loader


def get_eval_loader(data_dir: str, batch_size: int, num_workers: int = 8, eval_data: bool = True):
    """
    Returns a DataLoader for evaluation.
    If eval_data==False it reads training batches (data_batch_1..5) else test_batch.bin
    """
    if eval_data:
        files = ["test_batch.bin"]
    else:
        files = [
            "data_batch_1.bin",
            "data_batch_2.bin",
            "data_batch_3.bin",
            "data_batch_4.bin",
            "data_batch_5.bin",
        ]

    dataset = CIFAR10BinaryDataset(
        data_dir=data_dir,
        files=files,
        transform=_eval_transform(),
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    return loader


## Diagonal LSTM and Masked Convolution

20 Points

In [2]:
import logging
from typing import Optional, Tuple
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

logging.basicConfig(format="[%(asctime)s] %(message)s", datefmt="%m-%d %H:%M:%S")
logger = logging.getLogger(__name__)

DEFAULT_DATA_FORMAT = "NHWC"

def _is_nchw(x: torch.Tensor) -> bool:
    return x.dim() == 4 and x.shape[1] <= 4 and x.shape[2] > 4  # heuristic; not perfect

def ensure_nhwc(x: torch.Tensor) -> torch.Tensor:
    """Return tensor in NHWC layout (B,H,W,C)."""
    if x.dim() != 4:
        raise ValueError("Expect 4D tensor")
    # If likely NCHW (N,C,H,W) convert to NHWC
    if x.shape[1] <= 4 and x.shape[2] > 4:
        return x.permute(0, 2, 3, 1).contiguous()
    return x

def ensure_nchw(x: torch.Tensor) -> torch.Tensor:
    """Return tensor in NCHW layout (B,C,H,W)."""
    if x.dim() != 4:
        raise ValueError("Expect 4D tensor")
    # If likely NHWC convert to NCHW
    if x.shape[3] <= 4 and x.shape[1] > 4:
        return x.permute(0, 3, 1, 2).contiguous()
    if x.shape[1] <= 4 and x.shape[2] > 4:
        # ambiguous, assume NCHW already
        return x
    # assume NHWC (B,H,W,C)
    if x.shape[3] <= 4:
        return x.permute(0, 3, 1, 2).contiguous()
    return x

def get_shape(inputs: torch.Tensor):
    """Return python list-style shape similar to TF's .as_list()."""
    return list(inputs.size())

# ---------------------------
# skew / unskew (supports NHWC and NCHW; works on NHWC primarily)
# ---------------------------
def skew(inputs: torch.Tensor, scope: str = "skew") -> torch.Tensor:
    """
    Accepts NHWC ([B,H,W,C]) or NCHW ([B,C,H,W]) tensors.
    Returns tensor in the same layout, but with width = W + H - 1.
    """
    was_nchw = _is_nchw(inputs)
    # Work internally in NHWC
    if was_nchw:
        x = inputs.permute(0, 2, 3, 1).contiguous()
    else:
        x = inputs

    b, h, w, c = x.shape
    new_w = w + h - 1
    outputs = x.new_zeros(b, h, new_w, c)

    # Offset each row by its row index
    for row in range(h):
        outputs[:, row, row:row + w, :] = x[:, row, :, :]

    logger.debug(f"[skew] {scope} : {inputs.shape} -> {outputs.shape}")

    if was_nchw:
        return outputs.permute(0, 3, 1, 2).contiguous()
    return outputs


def unskew(inputs: torch.Tensor, width: Optional[int] = None, scope: str = "unskew") -> torch.Tensor:
    """
    Reverse of skew. Accepts NHWC or NCHW and returns in the same layout.
    """
    was_nchw = _is_nchw(inputs)
    # Work internally in NHWC
    if was_nchw:
        x = inputs.permute(0, 2, 3, 1).contiguous()
    else:
        x = inputs

    b, h, w_skew, c = x.shape
    if width is None:
        # Inverse of new_w = w + h - 1
        width = w_skew - h + 1

    outputs = x.new_zeros(b, h, width, c)
    for row in range(h):
        outputs[:, row, :, :] = x[:, row, row:row + width, :]

    logger.debug(f"[unskew] {scope} : {inputs.shape} -> {outputs.shape}")

    if was_nchw:
        return outputs.permute(0, 3, 1, 2).contiguous()
    return outputs




# ---------------------------
# conv2d with optional mask (mask type None/'A'/'B')
# ---------------------------
class MaskedConv2d(nn.Module):
    """
    A conv2d that mimics TF behavior with variable creation and masking.
    The module expects inputs either in NHWC or NCHW; internally we run conv in NCHW.
    weights_shape = [kernel_h, kernel_w, in_channels, out_channels] (TF style).
    """
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: Tuple[int, int],
                 mask_type: Optional[str] = None,
                 stride: Tuple[int, int] = (1, 1),
                 padding: str = "SAME",
                 bias: bool = True,
                 weights_initializer=None,   
                 name: str = "conv2d"):
        super().__init__()
        self.kernel_h, self.kernel_w = kernel_size
        self.mask_type = mask_type.lower() if mask_type is not None else None
        self.stride = stride
        self.padding = padding
        self.name = name

        # Create weight parameter in PyTorch conv format: [out_channels, in_channels, kh, kw]
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, self.kernel_h, self.kernel_w))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)

        # Initialize weights (Xavier uniform by default)
        nn.init.xavier_uniform_(self.weight)

        # Precompute mask if needed (create in TF ordering then transpose to PyTorch ordering)
        if self.mask_type is not None:
            mask = np.ones((self.kernel_h, self.kernel_w, in_channels, out_channels), dtype=np.float32)
            center_h = self.kernel_h // 2
            center_w = self.kernel_w // 2
            mask[center_h, center_w + 1:, :, :] = 0.
            mask[center_h + 1:, :, :, :] = 0.
            if self.mask_type == 'a':
                mask[center_h, center_w, :, :] = 0.
            # convert mask to shape [out, in, kh, kw] for direct multiplication with self.weight
            mask = mask.transpose(3, 2, 0, 1).copy()
            self.register_buffer('mask', torch.tensor(mask))
        else:
            self.mask = None

    def forward(self, inputs: torch.Tensor):
    # Accept NHWC or NCHW. Convert to NCHW for conv
        is_nchw = _is_nchw(inputs)
    
        if is_nchw:
            x = inputs
        else:
            # NHWC -> NCHW
            x = inputs.permute(0, 3, 1, 2).contiguous()
    
        weight = self.weight
        if self.mask is not None:
            weight = weight * self.mask

        if self.padding.upper() == "SAME":
            b, c, h, w = x.shape
            stride_h, stride_w = self.stride
    
            out_h = math.ceil(h / float(stride_h))
            out_w = math.ceil(w / float(stride_w))
    
            pad_h = max((out_h - 1) * stride_h + self.kernel_h - h, 0)
            pad_w = max((out_w - 1) * stride_w + self.kernel_w - w, 0)
    
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
    
            # F.pad uses (left, right, top, bottom)
            x = F.pad(x, (pad_left, pad_right, pad_top, pad_bottom))
            padding = 0
        else:
            padding = 0

        out = F.conv2d(x, weight, self.bias, stride=self.stride, padding=padding)
    
        # return in same layout as input
        if not is_nchw:
            out = out.permute(0, 2, 3, 1).contiguous()
    
        logger.debug(f"[conv2d_{self.mask_type}] {self.name} : {inputs.shape} -> {out.shape}")
        return out


# conv1d implemented via conv2d with kernel_w = 1
class MaskedConv1d(MaskedConv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=(1,1), padding="SAME", mask_type=None, bias=True, name="conv1d"):
        # kernel_size is int (height)
        super().__init__(in_channels=in_channels,
                         out_channels=out_channels,
                         kernel_size=(kernel_size, 1),
                         mask_type=mask_type,
                         stride=stride,
                         padding=padding,
                         bias=bias,
                         name=name)

# ---------------------------
# Diagonal LSTM Cell
# ---------------------------
class DiagonalLSTMCell(nn.Module):
    """
    Diagonal LSTM Cell equivalent converted from TF.
    - hidden_dims: number of hidden channels per spatial row
    - height: number of rows
    - channel: input channels (for conv1d s_to_s)
    """
    def __init__(self, hidden_dims: int, height: int, channel: int):
        super().__init__()
        self._hidden_dims = hidden_dims
        self._height = height
        self._channel = channel
        self._num_units = hidden_dims * height
        self._state_size = self._num_units * 2
        self._output_size = self._num_units

        # conv to compute s_to_s (2x1 conv as in TF conv1d)
        # We'll implement as a Conv2d with kernel (2,1) operating on [B, hidden_dims, height, 1]
        # But conv1d in TF used input channels = hidden_dims; output = 4*hidden_dims
        # Use groups=1
        self.s_to_s_conv = MaskedConv1d(in_channels=self._hidden_dims, out_channels=4*self._hidden_dims, kernel_size=2, padding="SAME", name='s_to_s')

    def forward(self, i_to_s: torch.Tensor, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        i_to_s: [B, 4 * height * hidden_dims]
        state: [B, 2 * num_units] where num_units = height * hidden_dims
        returns: h (B, height*hidden_dims), new_state (B, 2*num_units)
        """
        batch_size = i_to_s.size(0)
        num_units = self._num_units
    
        assert i_to_s.size(1) == 4 * num_units, \
            f"Expected i_to_s dim 1 = {4 * num_units}, got {i_to_s.size(1)}"
    
        # Split state into cell and hidden
        c_prev = state[:, :num_units]              # [B, H*D]
        h_prev = state[:, num_units:]             # [B, H*D]
    
        # Reshape hidden for state-to-state conv
        h_prev_reshaped = h_prev.view(batch_size, self._height, self._hidden_dims)  # [B, H, D]
    
        # Represent as NHWC for MaskedConv2d (so _is_nchw will treat it as NHWC)
        h_prev_nhwc = h_prev_reshaped.unsqueeze(2)  # [B, H, 1, D]
    
        # 2x1 conv along height dimension
        s_to_s = self.s_to_s_conv(h_prev_nhwc)      # [B, H, 1, 4*D] (NHWC)
        s_to_s = s_to_s.squeeze(2)                  # [B, H, 4*D]
    
        # Input-to-state term
        i_to_s_reshaped = i_to_s.view(batch_size, self._height, 4 * self._hidden_dims)  # [B, H, 4*D]

        # LSTM gates
        lstm_matrix = i_to_s_reshaped + s_to_s     # [B, H, 4*D]
        i, j, f, o = torch.chunk(lstm_matrix, 4, dim=2)  # each [B, H, D]
    
        c_prev_reshaped = c_prev.view(batch_size, self._height, self._hidden_dims)  # [B, H, D]
    
        forget_bias = 1.0
        new_c = torch.sigmoid(f + forget_bias) * c_prev_reshaped + torch.sigmoid(i) * torch.tanh(j)
        new_h = torch.tanh(new_c) * torch.sigmoid(o)
    
        h_flat = new_h.reshape(batch_size, -1)      # [B, H*D]
        new_state = torch.cat(
            [new_c.reshape(batch_size, -1), new_h.reshape(batch_size, -1)],
            dim=1
        )                                           # [B, 2*H*D]
    
        return h_flat, new_state


# ---------------------------
# diagonal_lstm (sequence loop)
# ---------------------------
def diagonal_lstm(inputs: torch.Tensor, conf, scope: str = 'diagonal_lstm') -> torch.Tensor:
    """
    inputs: NHWC or NCHW.

    Steps:
      - convert to NHWC if necessary
      - skew -> 1x1 conv to get input_to_state
      - iterate over skewed width with DiagonalLSTMCell
      - unskew back to original spatial layout

    conf must provide: hidden_dims (int), use_dynamic_rnn (optional, ignored here)
    Returns: outputs with channels = hidden_dims, in same layout as inputs.
    """
    was_nchw = _is_nchw(inputs)

    # Work in NHWC internally
    if was_nchw:
        x = inputs.permute(0, 2, 3, 1).contiguous()
    else:
        x = inputs

    b, h, w, c = x.shape
    hidden_dims = conf.hidden_dims

    # 1. Skew
    x_skew = skew(x, scope=scope + '_skew')         # [B, H, W_skew, C]
    _, h2, w_skew, c2 = x_skew.shape
    assert h2 == h

    # 2. Input-to-state: 1x1 conv to produce gates for LSTM
    i2s_conv = MaskedConv2d(
        in_channels=c2,
        out_channels=4 * hidden_dims,
        kernel_size=(1, 1),
        mask_type=None,
        padding="SAME",
        name=scope + '_i_to_s'
    )
    i_to_s = i2s_conv(x_skew)                      # [B, H, W_skew, 4*hidden_dims]

    # 3. Prepare sequence over width dimension
    rnn_inputs = i_to_s.permute(0, 2, 1, 3).contiguous()  # [B, W_skew, H, 4*hidden]
    rnn_inputs = rnn_inputs.view(b, w_skew, h * 4 * hidden_dims)  # [B, W_skew, H*4*hidden]

    # 4. RNN loop with DiagonalLSTMCell
    cell = DiagonalLSTMCell(hidden_dims=hidden_dims, height=h, channel=c2)
    num_units = hidden_dims * h
    state = x.new_zeros(b, 2 * num_units)

    outputs_steps = []
    for t in range(w_skew):
        h_t, state = cell(rnn_inputs[:, t, :], state)   # h_t: [B, H*hidden]
        outputs_steps.append(h_t)

    outputs_all = torch.stack(outputs_steps, dim=1)     # [B, W_skew, H*hidden]
    h_reshaped = outputs_all.view(b, w_skew, h, hidden_dims)
    h_reshaped = h_reshaped.permute(0, 2, 1, 3).contiguous()  # [B, H, W_skew, hidden]

    # 5. Unskew back to original width
    outputs = unskew(h_reshaped, width=w, scope=scope + '_unskew')  # [B, H, W, hidden]

    # Return in original layout
    if was_nchw:
        return outputs.permute(0, 3, 1, 2).contiguous()
    return outputs


# ---------------------------
# diagonal_bilstm
# ---------------------------
def diagonal_bilstm(inputs: torch.Tensor, conf, scope: str = 'diagonal_bilstm') -> torch.Tensor:
    """
    Compose forward and backward diagonal_lstm, then combine.
    If conf.use_residual is True, add a residual connection from inputs.
    Returns sum of forward and (shifted) backward outputs.
    """
    was_nchw = _is_nchw(inputs)

    # Work in NHWC internally
    if was_nchw:
        x = inputs.permute(0, 2, 3, 1).contiguous()
    else:
        x = inputs

    # Forward direction
    out_fw = diagonal_lstm(x, conf, scope=scope + '_fw')  # [B, H, W, D]

    # Backward direction: reverse along width
    x_rev = torch.flip(x, dims=[2])                       # [B, H, W, C]
    out_bw_rev = diagonal_lstm(x_rev, conf, scope=scope + '_bw')
    out_bw = torch.flip(out_bw_rev, dims=[2])             # align with forward ordering

    # Shift backward output down by one row to avoid seeing future pixels (as in the paper)
    b, h, w, d = out_bw.shape
    shifted_bw = out_bw.new_zeros(b, h, w, d)
    shifted_bw[:, 1:, :, :] = out_bw[:, :-1, :, :]        # top row is zero

    out = out_fw + shifted_bw

    # Optional residual connection
    if getattr(conf, 'use_residual', False):
        if x.shape[-1] == out.shape[-1]:
            # Simple residual if channel dims match
            out = out + x
        else:
            # 1x1 conv to match channels
            res_conv = MaskedConv2d(
                in_channels=x.shape[-1],
                out_channels=out.shape[-1],
                kernel_size=(1, 1),
                mask_type=None,
                padding="SAME",
                name=scope + '_res'
            )
            res = res_conv(x)
            out = out + res

    # Return in original layout
    if was_nchw:
        return out.permute(0, 3, 1, 2).contiguous()
    return out



In [3]:
!pip install "protobuf<5" --upgrade
!pip install "tensorboard<2.16" --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 6.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 4.25.8 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatibl

In [4]:
import os
import glob
import logging
from typing import Optional, Dict, Any, Iterable

import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter

logger = logging.getLogger(__name__)
logging.basicConfig(format="[%(asctime)s] %(message)s", datefmt="%m-%d %H:%M:%S")


class Statistic:
    """
    the Statistic helper.

    Args:
        model: torch.nn.Module to save/load state_dict from.
        optimizer: optional torch.optim.Optimizer to save/load state.
        data_tag: str label used in TensorBoard scalar names .
        model_dir: directory where checkpoints and logs will be written.
        test_step: not used internally here but kept for API similarity (you can use it externally).
        max_to_keep: how many checkpoints to keep; older are deleted.
        device: device to load tensors to when loading checkpoints (default: cpu).
    """
    def __init__(
        self,
        model: nn.Module,
        optimizer: Optional[torch.optim.Optimizer],
        data_tag: str,
        model_dir: str,
        test_step: int,
        max_to_keep: int = 20,
        device: Optional[torch.device] = None,
    ):
        self.model = model
        self.optimizer = optimizer
        self.data_tag = data_tag
        self.model_dir = model_dir
        self.test_step = test_step
        self.max_to_keep = max_to_keep
        self.device = device if device is not None else torch.device("cpu")

        # internal counter t
        self.t = 0

        # tensorboard writer
        log_dir = os.path.join("logs", self.model_dir)
        os.makedirs(log_dir, exist_ok=True)
        self.writer = SummaryWriter(log_dir=log_dir)

        # Ensure model_dir exists for checkpoints
        os.makedirs(self.model_dir, exist_ok=True)

        # Optionally keep a list of saved checkpoints on disk (sorted)
        self._refresh_checkpoint_list()

        # call reset to initialize whatever you want (kept for API parity)
        self.reset()

        logger.info("Statistic initialized. logs -> %s, checkpoints -> %s", log_dir, self.model_dir)

    def reset(self):
        """
        Override or extend if you need to accumulate running statistics.
        """
        # user can extend this method if they want to keep rolling averages, etc.
        pass

    def _refresh_checkpoint_list(self):
        """
        Internal helper: update the cached list of checkpoint paths sorted by step (ascending).
        """
        pattern = os.path.join(self.model_dir, "checkpoint_*.pt")
        ckpts = glob.glob(pattern)
        # parse t from filename checkpoint_{t}.pt
        def _t_from_name(p):
            try:
                base = os.path.basename(p)
                t_str = base.replace("checkpoint_", "").replace(".pt", "")
                return int(t_str)
            except Exception:
                return -1
        ckpts_sorted = sorted(ckpts, key=_t_from_name)
        self._checkpoints = ckpts_sorted

    def _prune_checkpoints(self):
        """
        Keep only the last `max_to_keep` checkpoints, remove older ones.
        """
        self._refresh_checkpoint_list()
        if self.max_to_keep is None:
            return
        while len(self._checkpoints) > self.max_to_keep:
            old = self._checkpoints.pop(0)
            try:
                os.remove(old)
                logger.info("Removed old checkpoint: %s", old)
            except OSError:
                logger.warning("Failed to remove old checkpoint: %s", old)

    def on_step(self, train_l: float, test_l: float):
        """
        Called at end of a step (or epoch) to increment counter, write summaries, save checkpoints and reset accumulators.
        """
        # increment counter first
        self.t += 1

        # write summaries
        self.inject_summary({'train_l': train_l, 'test_l': test_l}, self.t)

        # save model
        self.save_model(self.t)

        # reset accumulators if any
        self.reset()

    def get_t(self) -> int:
        """Return current step counter."""
        return int(self.t)

    def inject_summary(self, tag_dict: Dict[str, float], t: Optional[int] = None):
        """
        Write scalar summaries to TensorBoard.
        tag_dict: mapping of tag->value, e.g. {'train_l': 0.5}
        t: step (if None uses current internal t)
        """
        step = self.t if t is None else int(t)
        for tag, value in tag_dict.items():
            full_tag = f"{self.data_tag}/{tag}"
            # ensure value is a python float
            try:
                val = float(value)
            except Exception:
                val = float(value.item()) if hasattr(value, "item") else float(value)
            self.writer.add_scalar(full_tag, val, step)
        # flush may be helpful to make summary available quickly
        self.writer.flush()

    def save_model(self, t: Optional[int] = None, extra_state: Optional[Dict[str, Any]] = None):
        """
        Save checkpoint for model (+ optimizer if supplied) and the internal counter 't'.
        extra_state: optional mapping of additional items to store (e.g., scheduler state).
        """
        step = self.t if t is None else int(t)
        ckpt_name = os.path.join(self.model_dir, f"checkpoint_{step}.pt")
        state = {
            't': step,
            'model_state': self.model.state_dict()
        }
        if self.optimizer is not None:
            state['optimizer_state'] = self.optimizer.state_dict()
        if extra_state:
            state['extra'] = extra_state

        # save atomically
        tmp_name = ckpt_name + ".tmp"
        torch.save(state, tmp_name)
        os.replace(tmp_name, ckpt_name)
        logger.info("Saved checkpoint: %s", ckpt_name)

        # refresh and prune
        self._refresh_checkpoint_list()
        self._prune_checkpoints()        

    def load_model(self, checkpoint_path: Optional[str] = None, map_location: Optional[torch.device] = None) -> bool:
        """
        Load the latest checkpoint (if checkpoint_path is None) or load the provided checkpoint file.
        Returns True if load succeeded, False otherwise.
        map_location: device to map the checkpoint tensors onto (default: self.device)
        """
 
        map_location = map_location or self.device

        if checkpoint_path is None:
            # find latest checkpoint
            self._refresh_checkpoint_list()
            if not self._checkpoints:
                logger.info("No checkpoints found in %s", self.model_dir)
                return False
            checkpoint_path = self._checkpoints[-1]

        if not os.path.exists(checkpoint_path):
            logger.warning("Checkpoint path does not exist: %s", checkpoint_path)
            return False

        logger.info("Loading checkpoint: %s", checkpoint_path)
        ckpt = torch.load(checkpoint_path, map_location=map_location)

        # load model state
        if 'model_state' in ckpt:
            try:
                self.model.load_state_dict(ckpt['model_state'])
            except Exception as e:
                logger.exception("Failed to load model_state: %s", e)
                return False
        else:
            logger.warning("Checkpoint missing 'model_state' key")

        # load optimizer state if present and optimizer available
        if 'optimizer_state' in ckpt and self.optimizer is not None:
            try:
                self.optimizer.load_state_dict(ckpt['optimizer_state'])
            except Exception as e:
                logger.exception("Failed to load optimizer_state: %s", e)
                # continue, not fatal

        # restore t if present
        if 't' in ckpt:
            self.t = int(ckpt['t'])
        else:
            # try to infer from filename
            base = os.path.basename(checkpoint_path)
            try:
                self.t = int(base.replace("checkpoint_", "").replace(".pt", ""))
            except Exception:
                self.t = 0

        logger.info("Load SUCCESS: %s (t=%d)", checkpoint_path, int(self.t))
        return True


    def close(self):
        """Close writer etc."""
        try:
            self.writer.close()
        except Exception:
            pass


2025-11-20 20:15:42.294743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763669742.491513      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763669742.550240      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
import logging
logging.basicConfig(format="[%(asctime)s] %(message)s", datefmt="%m-%d %H:%M:%S")

import os
import sys
import pprint
import tarfile
import hashlib
import datetime
import numpy as np

from types import SimpleNamespace
from typing import Any, Dict, Iterable, Optional

# Python3 urllib
import urllib.request

from PIL import Image

pp = pprint.PrettyPrinter().pprint
logger = logging.getLogger(__name__)

# -------------------------
# Small helpers
# -------------------------
def mprint(matrix: Iterable[Iterable[float]], pivot: float = 0.5):
    """Pretty-print a binary-style matrix using '#' and ' ' like the original."""
    for array in matrix:
        print("".join("#" if i > pivot else " " for i in array))


def get_timestamp() -> str:
    """Return a filesystem-friendly timestamp string with local timezone info."""
    now = datetime.datetime.now().astimezone()
    return now.strftime('%Y_%m_%d_%H_%M_%S')


def binarize(images):
    """
    Binarize image(s) by sampling from Uniform(0,1) < pixel_value.
    Accepts a NumPy array or a PyTorch tensor. Returns same type as input (numpy or torch.FloatTensor).
    """
    try:
        import torch
        is_torch = torch.is_tensor(images)
    except Exception:
        is_torch = False

    if is_torch:
        # produce a tensor of same shape & device
        rand = torch.rand_like(images)
        return (rand < images).float()
    else:
        # assume numpy
        return (np.random.uniform(size=images.shape) < images).astype('float32')


# -------------------------
# Image saving
# -------------------------
def save_images(images, height: int, width: int, n_row: int, n_col: int,
                cmin: float = 0.0, cmax: float = 1.0, directory: str = "./", prefix: str = "sample"):
    """
    Save a grid of images to a single image file.
    - images: numpy array of shape (n_row*n_col, H, W) or (n_row*n_col, H, W, C)
              OR shaped (n_row, n_col, H, W) etc. This function will attempt to reshape sensibly.
    - height, width: single image H,W
    - n_row, n_col: grid layout
    """
    # convert torch -> numpy if needed
    try:
        import torch
        if torch.is_tensor(images):
            images = images.detach().cpu().numpy()
    except Exception:
        pass

    imgs = np.asarray(images)
    # handle shapes:
    # If flat list of images: (N, H, W) or (N, H, W, C)
    if imgs.ndim == 3:
        # grayscale stack
        N, H, W = imgs.shape
        C = 1
        imgs = imgs.reshape((N, H, W))
        imgs = imgs.reshape((n_row, n_col, H, W))
        imgs = imgs.transpose(1, 2, 0, 3).reshape((H * n_row, W * n_col))
        mode = 'L'
    elif imgs.ndim == 4:
        N, H, W, C = imgs.shape
        if C == 1:
            imgs = imgs.reshape((n_row, n_col, H, W))
            imgs = imgs.transpose(1, 2, 0, 3).reshape((H * n_row, W * n_col))
            mode = 'L'
        elif C == 3:
            # arrange into grid with channels last
            imgs = imgs.reshape((n_row, n_col, H, W, C))
            imgs = imgs.transpose(1, 2, 0, 3, 4).reshape((H * n_row, W * n_col, C))
            mode = 'RGB'
        else:
            # unsupported channel count: collapse or take first channel
            imgs = imgs[..., 0]
            imgs = imgs.reshape((n_row, n_col, H, W))
            imgs = imgs.transpose(1, 2, 0, 3).reshape((H * n_row, W * n_col))
            mode = 'L'
    else:
        raise ValueError("Unsupported image array shape: %s" % (imgs.shape,))

    # scale pixels from [cmin,cmax] to [0,255]
    if mode == 'L':
        norm = (imgs - cmin) / max((cmax - cmin), 1e-8)
        arr = np.clip(norm * 255.0, 0, 255).astype(np.uint8)
        pil_img = Image.fromarray(arr, mode='L')
    else:
        norm = (imgs - cmin) / max((cmax - cmin), 1e-8)
        arr = np.clip(norm * 255.0, 0, 255).astype(np.uint8)
        pil_img = Image.fromarray(arr, mode='RGB')

    filename = f'{prefix}_{get_timestamp()}.jpg'
    out_path = os.path.join(directory, filename)
    os.makedirs(directory, exist_ok=True)
    pil_img.save(out_path)
    logger.info("Saved image grid to %s", out_path)


# -------------------------
# Model / config helpers
# -------------------------
def get_model_dir(config, exceptions=None):
    """
    Robust replacement for TF-specific get_model_dir:
      - Accepts TF flags object (which stores flags under __flags) or argparse/Namespace/SimpleNamespace.
      - Builds a readable model dir name from config key/value pairs excluding `exceptions`.
      - If the generated name is too long, falls back to an md5 hash to keep path sane.
    """
    exceptions = set(exceptions or [])
    # Try TF-style flags first
    try:
        if hasattr(config, '__dict__') and '__flags' in config.__dict__:
            attrs = dict(config.__dict__['__flags'])
        else:
            # argparse.Namespace / SimpleNamespace / plain object
            attrs = dict(vars(config))
    except Exception:
        # as a last resort, try to use __dict__
        try:
            attrs = dict(config.__dict__)
        except Exception:
            attrs = {}

    # Filter out exceptions and None/empty values that are not informative
    items = []
    for k in sorted(attrs.keys()):
        if k in exceptions:
            continue
        if k.startswith('_'):
            continue
        v = attrs[k]
        if callable(v):
            vstr = v.__name__
        else:
            try:
                vstr = str(v)
            except Exception:
                vstr = repr(v)
        # shorten long values
        if len(vstr) > 40:
            vstr = hashlib.md5(vstr.encode('utf-8')).hexdigest()[:8]
        items.append(f"{k}={vstr}")

    if not items:
        name = "default"
    else:
        name = "_".join(items)

    # sanitize name (remove spaces, slashes)
    name = name.replace(" ", "").replace("/", "_").replace("\\", "_")

    # If too long, shorten using md5
    if len(name) > 180:
        name = hashlib.md5(name.encode("utf-8")).hexdigest()

    return os.path.join('checkpoints', name) + '/'


def preprocess_conf(conf):
    """
    Placeholder to keep API parity with your previous code.
    If you need to normalize or canonicalize flags/options, do it here.
    """
    # For argparse/Namespace we don't need to do anything by default.
    return conf


def check_and_create_dir(directory: str):
    """Create dir if not exists (logs)"""
    if not os.path.exists(directory):
        logger.info('Creating directory: %s' % directory)
        os.makedirs(directory)
    else:
        logger.info('Skip creating directory: %s' % directory)


def show_all_variables(model: Optional[Any] = None):
    """
    Print all trainable variables.
    If `model` is provided (torch.nn.Module), iterate its parameters.
    """
    try:
        import torch
        if model is None:
            logger.warning("No model passed to show_all_variables(model). Nothing to show.")
            return
        total_count = 0
        for idx, (name, param) in enumerate(model.named_parameters()):
            shape = tuple(param.shape)
            count = int(np.prod(shape))
            print("[%2d] %s %s = %s" % (idx, name, shape, count))
            total_count += count
        print("[Total] variable size: %s" % "{:,}".format(total_count))
    except Exception as e:
        logger.exception("show_all_variables failed: %s", e)

## Network
in this part we will impelement a neural network that can be configured to perform pixel-wise predictions, specifically for image generation tasks such as those in PixelCNN or PixelRNN models. This class supports both training and inference, and can generate outputs pixel-by-pixel in a raster scan order, leveraging various types of convolutions (standard and masked) and recurrent layers (e.g., LSTM).

25 Points

In [6]:
class Network(nn.Module):
    """
    PyTorch Network class.
    """
    def __init__(self, conf, height: int, width: int, channel: int, device: Optional[torch.device] = None):
        super().__init__()
        logger.info("Building %s starts!" % conf.model)
        self.conf = conf
        self.data = conf.data
        self.height, self.width, self.channel = height, width, channel
        self.device = device if device is not None else (torch.device("cuda") if conf.use_gpu and torch.cuda.is_available() else torch.device("cpu"))

        # Data format decisions
        self.data_format = "NHWC" if conf.use_gpu else "NCHW"

        # Build layers - FIXED CHANNEL DIMENSIONS
        in_channels = channel

        # Calculate conv_out_channels based on residual connections
        if conf.use_residual and conf.model == "pixel_rnn":
            conv_out_channels = conf.hidden_dims * 2
        else:
            conv_out_channels = conf.hidden_dims

        # Input convolution
        self.conv_inputs = nn.Conv2d(in_channels, conv_out_channels, kernel_size=7, padding=3)

        # Build recurrent/conv layers
        self.recurrent_length = conf.recurrent_length
        self.out_recurrent_length = conf.out_recurrent_length

        if conf.model == "pixel_cnn":
            # For pixel_cnn, use masked convolutions with proper channel dimensions
            self.conv_blocks = nn.ModuleList()
            for idx in range(self.recurrent_length):
                in_ch = conv_out_channels if idx == 0 else conf.hidden_dims
                # Use regular conv2d for now - replace with masked conv if available
                self.conv_blocks.append(nn.Conv2d(in_ch, conf.hidden_dims, kernel_size=3, padding=1))
        else:
            # For pixel_rnn we will call diagonal_bilstm in forward()
            self.conv_blocks = None

        # Output recurrent layers (1x1 convs + ReLU)
        self.out_convs = nn.ModuleList()
        for idx in range(self.out_recurrent_length):
            in_ch = conf.hidden_dims if idx == 0 else conf.out_hidden_dims
            self.out_convs.append(nn.Conv2d(in_ch, conf.out_hidden_dims, kernel_size=1))

        # Final logits conv - FIXED: output should match input channels for binary prediction
        if channel == 1:
            # Single output logit per pixel
            in_ch = conf.out_hidden_dims if self.out_recurrent_length > 0 else conv_out_channels
            self.conv2d_out_logits = nn.Conv2d(in_ch, 1, kernel_size=1)
            self.loss_fn = nn.BCEWithLogitsLoss(reduction='mean')
        else:
            # For RGB/categorical (not implemented in original)
            raise NotImplementedError("RGB branch not implemented (same as original).")

        # Optimizer (RMSProp) and grad clipping
        self.optimizer = optim.RMSprop(self.parameters(), lr=conf.learning_rate)
        self.grad_clip = conf.grad_clip

        # Move to device
        self.to(self.device)
        logger.info("Building %s finished!" % conf.model)

    def _to_nchw(self, x: torch.Tensor) -> torch.Tensor:
        """Convert incoming tensor to NCHW for internal conv ops."""
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
        x = x.to(self.device)
        if x.dim() != 4:
            raise ValueError("Input must be 4D tensor")
        if self.data_format == "NHWC":
            # [B,H,W,C] -> [B,C,H,W]
            x = x.permute(0, 3, 1, 2).contiguous()
        return x.float()

    def _to_output_layout(self, x: torch.Tensor) -> torch.Tensor:
        """Convert internal outputs (NCHW) back to requested external layout."""
        if self.data_format == "NHWC":
            return x.permute(0, 2, 3, 1).contiguous()
        return x

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Forward pass: from input image to per-pixel logits."""
        # Convert to NCHW and move to device
        x = self._to_nchw(inputs)  # [B, C, H, W]

        # Initial 7x7 conv
        h = self.conv_inputs(x)
        h = F.relu(h)

        if self.conf.model == "pixel_cnn":
            # Stack of (masked) conv blocks
            for conv in self.conv_blocks:
                h = conv(h)
                h = F.relu(h)
        else:
            # PixelRNN: diagonal BiLSTM over spatial positions
            # diagonal_bilstm expects NHWC for hidden feature maps
            h_nhwc = h.permute(0, 2, 3, 1).contiguous()  # [B, H, W, C]
            h_nhwc = diagonal_bilstm(h_nhwc, self.conf, scope='diagonal_bilstm')
            # Convert back to NCHW for 1x1 convs
            h = h_nhwc.permute(0, 3, 1, 2).contiguous()  # [B, hidden_dims, H, W]

        # Output 1x1 conv stack
        for conv in self.out_convs:
            h = conv(h)
            h = F.relu(h)

        # Final logits per pixel (no activation)
        logits = self.conv2d_out_logits(h)  # [B, 1, H, W]

        # Return in external layout
        return self._to_output_layout(logits)

    def predict(self, images):
        """Predict probabilities p(x=1 | context) for given images."""
        self.eval()
        with torch.no_grad():
            logits = self.forward(images)          # same layout as inputs (NCHW or NHWC)
            probs_t = torch.sigmoid(logits)        # convert logits -> probabilities
        return probs_t.detach().cpu().numpy()

    def test(self, images, with_update: bool = False):
        """
        Compute loss for images, optionally update weights.
        images: batch of binary images (same layout as network's external layout).
        """
        if with_update:
            self.train()
        else:
            self.eval()

        # Forward
        if with_update:
            logits = self.forward(images)          # tracked graph
        else:
            with torch.no_grad():
                logits = self.forward(images)

        # Convert logits and targets to common NCHW layout for loss computation
        logits_nchw = self._to_nchw(logits)        # [B, 1, H, W]
        targets_nchw = self._to_nchw(images)       # [B, 1, H, W]

        loss = self.loss_fn(logits_nchw, targets_nchw)

        if with_update:
            self.optimizer.zero_grad()
            loss.backward()
            if self.grad_clip is not None and self.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(self.parameters(), self.grad_clip)
            self.optimizer.step()

        return float(loss.detach().cpu().item())

    def generate(self, batch_size: int = 100):
        """
        Generate binary samples in raster order (top-left -> bottom-right).
        Uses the current network as an autoregressive model by repeatedly
        feeding back its own samples.
        """
        self.eval()

        # Create empty canvas of zeros
        if self.data_format == "NHWC":
            # [B, H, W, C]
            samples = torch.zeros(
                (batch_size, self.height, self.width, self.channel),
                device=self.device,
                dtype=torch.float32,
            )
        else:
            # [B, C, H, W]
            samples = torch.zeros(
                (batch_size, self.channel, self.height, self.width),
                device=self.device,
                dtype=torch.float32,
            )

        with torch.no_grad():
            for i in range(self.height):
                for j in range(self.width):
                    # Compute logits for current partially filled samples
                    logits = self.forward(samples)  # external layout

                    if self.data_format == "NHWC":
                        # logits: [B, H, W, 1]
                        logits_ij = logits[:, i, j, 0]         # [B]
                    else:
                        # logits: [B, 1, H, W]
                        logits_ij = logits[:, 0, i, j]         # [B]

                    probs_ij = torch.sigmoid(logits_ij)        # [B]
                    # Sample Bernoulli for this pixel
                    new_vals = torch.bernoulli(probs_ij)       # [B]

                    # Write sampled value into canvas
                    if self.data_format == "NHWC":
                        samples[:, i, j, 0] = new_vals
                    else:
                        samples[:, 0, i, j] = new_vals

        return samples.detach().cpu().numpy()


## Main

10 Points

In [7]:
import os
import logging
import argparse
import math
from types import SimpleNamespace
import torch.optim as optim

import numpy as np
from tqdm import trange

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

     # <-- adjust path if needed

# logging
logging.basicConfig(format="[%(asctime)s] %(message)s", datefmt="%m-%d %H:%M:%S")
logger = logging.getLogger()
# default level set later by args


def parse_args():
    parser = argparse.ArgumentParser()
    # network
    parser.add_argument("--model", type=str, default="pixel_cnn", help="name of model [pixel_rnn, pixel_cnn]")
    parser.add_argument("--batch_size", type=int, default=100, help="size of a batch")
    parser.add_argument("--hidden_dims", type=int, default=16, help="dimesion of hidden states of LSTM or Conv layers")
    parser.add_argument("--recurrent_length", type=int, default=7, help="the length of LSTM or Conv layers")
    parser.add_argument("--out_hidden_dims", type=int, default=32, help="dimesion of hidden states of output Conv layers")
    parser.add_argument("--out_recurrent_length", type=int, default=2, help="the length of output Conv layers")
    parser.add_argument("--use_residual", action="store_true", default=False, help="whether to use residual connections or not")

    # training
    parser.add_argument("--max_epoch", type=int, default=100000, help="# of steps (total) to run")
    parser.add_argument("--test_step", type=int, default=100, help="# of steps between tests")
    parser.add_argument("--save_step", type=int, default=1000, help="# of steps between saves/sampling")
    parser.add_argument("--learning_rate", type=float, default=1e-3, help="learning rate")
    parser.add_argument("--grad_clip", type=float, default=1.0, help="value of gradient to be used for clipping")
    parser.add_argument("--use_gpu", action="store_true", default=True, help="whether to use gpu for training")

    # data
    parser.add_argument("--data", type=str, default="cifar", help="name of dataset [mnist, cifar]")
    parser.add_argument("--data_dir", type=str, default="data", help="name of data directory")
    parser.add_argument("--sample_dir", type=str, default="samples", help="name of sample directory")

    # debug / misc
    parser.add_argument("--is_train", action="store_true", default=True, help="training or testing")
    parser.add_argument("--display", action="store_true", default=False, help="whether to display the training results or not")
    parser.add_argument("--log_level", type=str, default="INFO", help="log level [DEBUG, INFO, WARNING, ERROR, CRITICAL]")
    parser.add_argument("--random_seed", type=int, default=123, help="random seed for python")

    # In notebook / script mode, we pass an empty list so it doesn't parse sys.argv
    return parser.parse_args([])


def main():
    args = parse_args()
    # convert argparse Namespace to a conf-like SimpleNamespace (so older code expecting attributes works)
    conf = SimpleNamespace(**vars(args))

    # logging level
    logger.setLevel(conf.log_level)

    # random seeds
    np.random.seed(conf.random_seed)
    torch.manual_seed(conf.random_seed)
    if conf.use_gpu and torch.cuda.is_available():
        torch.cuda.manual_seed_all(conf.random_seed)

    # Build model dir and preprocess config if helper exists
    model_dir = get_model_dir(
        conf,
        [
            "data_dir",
            "sample_dir",
            "max_epoch",
            "test_step",
            "save_step",
            "is_train",
            "random_seed",
            "log_level",
            "display",
        ],
    )
    preprocess_conf(conf)

    DATA_DIR = os.path.join(conf.data_dir, conf.data)
    SAMPLE_DIR = os.path.join(conf.sample_dir, conf.data, model_dir)

    check_and_create_dir(DATA_DIR)
    check_and_create_dir(SAMPLE_DIR)

    device = torch.device("cuda" if (conf.use_gpu and torch.cuda.is_available()) else "cpu")
    logger.info("Using device: %s", device)

    # 0. prepare datasets + network

    if conf.data == "cifar":
        # Transform: RGB -> Grayscale (1 channel) -> Tensor
        # If using GPU (NHWC mode), also convert from [C,H,W] -> [H,W,C]
        tfms = [
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
        ]

        if conf.use_gpu and torch.cuda.is_available():
            # Convert CHW -> HWC so that external layout is NHWC
            class ToNHWC(object):
                def __call__(self, x: torch.Tensor) -> torch.Tensor:
                    # x: [C,H,W] -> [H,W,C]
                    return x.permute(1, 2, 0)

            tfms.append(ToNHWC())

        transform = transforms.Compose(tfms)

        # CIFAR-10 train / test datasets
        train_dataset = datasets.CIFAR10(
            root=DATA_DIR,
            train=True,
            download=True,
            transform=transform,
        )
        test_dataset = datasets.CIFAR10(
            root=DATA_DIR,
            train=False,
            download=True,
            transform=transform,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=conf.batch_size,
            shuffle=True,
            drop_last=True,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=conf.batch_size,
            shuffle=False,
            drop_last=False,
        )

        # Infer height, width, channel from one sample
        sample_x, _ = train_dataset[0]
        if not torch.is_tensor(sample_x):
            sample_x = torch.from_numpy(sample_x)

        if conf.use_gpu and torch.cuda.is_available():
            # NHWC: [H, W, C]
            height, width, channel = sample_x.shape
        else:
            # NCHW: [C, H, W]
            channel, height, width = sample_x.shape

        # Build network
        network = Network(conf, height, width, channel, device=device)

    else:
        raise ValueError("Unknown dataset: %s" % conf.data)

    # 1. Statistic helper (for logging + checkpoints)
    try:
        stat = Statistic(network, network.optimizer, conf.data, model_dir, conf.test_step)
    except TypeError:
        # older Statistic signature variations: try with optimizer attr
        stat = Statistic(
            network,
            network.optimizer if hasattr(network, "optimizer") else None,
            conf.data,
            model_dir,
            conf.test_step,
        )

    # Try loading existing checkpoint
    stat.load_model()

    if conf.is_train:
        logger.info("Training starts!")
        initial_step = stat.get_t() if stat else 0

        # Global step counter starts from existing t to support resume
        iterator = trange(initial_step, conf.max_epoch, ncols=70)
        train_iter = iter(train_loader)
        last_test_loss = float("nan")

        for step in iterator:
            global_step = step + 1

            # 1. train: one batch
            try:
                train_images, _ = next(train_iter)
            except StopIteration:
                # restart epoch over train data
                train_iter = iter(train_loader)
                train_images, _ = next(train_iter)

            train_loss = network.test(train_images, with_update=True)

            # 2. test (every conf.test_step steps)
            if global_step % conf.test_step == 0:
                test_losses = []
                for test_images, _ in test_loader:
                    loss_val = network.test(test_images, with_update=False)
                    test_losses.append(loss_val)
                if test_losses:
                    last_test_loss = float(np.mean(test_losses))
                logger.info(
                    "Step %d | train_loss=%.4f | test_loss=%.4f",
                    global_step,
                    train_loss,
                    last_test_loss,
                )

            # 3. generate samples + checkpoint/logging (every conf.save_step steps)
            if global_step % conf.save_step == 0:
                # Log + save model via Statistic helper
                stat.on_step(train_l=train_loss, test_l=last_test_loss)

                # Generate samples and save as image grid
                try:
                    samples = network.generate(batch_size=conf.batch_size)
                    samples_t = torch.from_numpy(samples).float()

                    # Convert to NCHW for saving
                    if conf.use_gpu and torch.cuda.is_available():
                        # samples: [B, H, W, C] -> [B, C, H, W]
                        if samples_t.dim() == 4 and samples_t.shape[-1] in (1, 3):
                            samples_t = samples_t.permute(0, 3, 1, 2)

                    samples_t = samples_t.clamp(0.0, 1.0)

                    # Build a nice grid
                    n_show = min(samples_t.size(0), 64)
                    nrow = int(math.sqrt(n_show))
                    grid = make_grid(samples_t[:n_show], nrow=nrow, padding=2)

                    out_path = os.path.join(SAMPLE_DIR, f"samples_step_{global_step}.png")
                    save_image(grid, out_path)
                    logger.info("Saved samples to %s", out_path)
                except Exception as e:
                    logger.warning("Failed to save samples: %s", e)

    else:
        logger.info("Evaluation / sampling mode (is_train=False) not fully implemented here.")
        # You can add a simple eval / generate-only block if needed:
        # e.g., run test_loader through network.test(..., with_update=False)
        # and/or call network.generate(...) and save samples.


if __name__ == "__main__":
    main()


[11-20 20:16:16] Creating directory: data/cifar
[11-20 20:16:16] Creating directory: samples/cifar/checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/
[11-20 20:16:16] Using device: cuda
100%|██████████| 170M/170M [00:02<00:00, 57.1MB/s] 
[11-20 20:16:22] Building pixel_cnn starts!
[11-20 20:16:22] Building pixel_cnn finished!
[11-20 20:16:22] Statistic initialized. logs -> logs/checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/, checkpoints -> checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/
[11-20 20:16:22] No checkpoints found in checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/
[11-20 20:16:22] Training starts!
  1%|▎                           | 995/100000 [00:40<39:33, 41.71it/s][11-20 20:17:04] Step 1000 | train_loss=0.5830 | test_loss=0.5678
[11-20 20:17:04] Saved checkpoint: checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/checkpoint_1.pt
[11-20 20:17:09] Saved samples to samples/cifar/checkpoints/1bab95bd8fa301e6c14740b36cc7f5c6/samples_step_1000.png
  2%|▌                          | 1997/100000 [0